# Análise da Composição do Universo de Investimento e Portfólios
Este notebook realiza a análise da composição do universo de ativos utilizado para construção dos portfólios, separando:
1. Ações (Stocks)
2. Não-Ações (ETFs, Closed-End Funds, REITs, etc.)

Todas as visualizações são apresentadas em formato tabular (Tabelas com gradiente de cor para facilitar a leitura), mostrando sempre duas visões:
- **Percentual por Quantidade de Ativos (Count %)**
- **Percentual por Market Cap (Mcap %)**


In [31]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display

# Função auxiliar para exibir tabelas com estilo
def display_styled_table(df, title):
    print(f"\n--- {title} ---")
    styled_df = (df.style
                 .format("{:.2f}%")
                 .background_gradient(cmap='Blues', axis=1))
    display(styled_df)


## 1. Carregamento e Preparação dos Dados


In [32]:
df_hcm = pd.read_parquet("../../data/07_portfolios_metadata/complete_metadata_hcm.parquet")
df_pozzi = pd.read_parquet("../../data/07_portfolios_metadata/complete_metadata_pozzi.parquet")

print(f"Shape HCM: {df_hcm.shape}")
print(f"Shape Pozzi: {df_pozzi.shape}")

# Extraindo Market Cap baseado no ano de cada linha
def extract_mcap(df):
    def get_mcap(row):
        year = int(row['year'])
        col = f'mcap_{year}'
        if col in row.index:
            return row[col]
        return np.nan
    df['mcap'] = df.apply(get_mcap, axis=1)
    return df

df_hcm = extract_mcap(df_hcm)
df_pozzi = extract_mcap(df_pozzi)


Shape HCM: (40662, 38)
Shape Pozzi: (40662, 38)


In [34]:
# --- CORREÇÃO DE OUTLIER ---
# Copiando os dados corretos de Market Cap da BRK-B para a BRK-A
for df in [df_hcm, df_pozzi]:
    brk_b_mcaps = df[df['Ticker'] == 'BRK-B'].set_index('year')['mcap']
    
    def fix_brka(row):
        if row['Ticker'] == 'BRK-A' and row['year'] in brk_b_mcaps.index:
            return brk_b_mcaps.loc[row['year']]
        return row['mcap']
        
    df['mcap'] = df.apply(fix_brka, axis=1)
# ---------------------------

## 2. Classificação: Ações vs Não-Ações (ETFs, CEFs, REITs)


In [35]:
def classify_asset_type(row):
    industry = str(row['Industry']).lower()
    sector = str(row['Sector']).lower()
    
    if 'exchange traded fund' in industry or 'etf' in industry:
        return 'ETF'
    elif 'closed-end fund' in industry:
        return 'Closed-End Fund'
    elif 'reit' in industry or 'reit' in sector:
        return 'REIT'
    elif 'mutual fund' in industry or 'fund' in industry:
        return 'Other Fund'
    else:
        return 'Stock'

for df in [df_hcm, df_pozzi]:
    df['Asset_Class'] = df.apply(classify_asset_type, axis=1)
    df['Main_Type'] = df['Asset_Class'].apply(lambda x: 'Stock' if x == 'Stock' else 'Non-Stock')

unique_tickers = df_hcm["Ticker"].unique()
df_hcm_unique = df_hcm.iloc[:len(unique_tickers)]

print("Distribuição Global de Tipos de Ativos (Contagem):")
display(pd.DataFrame(df_hcm_unique['Asset_Class'].value_counts()))

Distribuição Global de Tipos de Ativos (Contagem):


,count
Asset_Class,
Stock,3186
ETF,1053
Closed-End Fund,472
REIT,183


In [36]:
len(unique_tickers)

4894

## 3. Análise da Composição do Universo Ano a Ano


In [37]:
def analyze_composition(df, group_col, columns_col):
    # Por Quantidade
    count_df = df.groupby([group_col, columns_col]).size().unstack(fill_value=0)
    count_pct = count_df.div(count_df.sum(axis=1), axis=0) * 100
    
    # Por Market Cap
    mcap_df = df.groupby([group_col, columns_col])['mcap'].sum().unstack(fill_value=0)
    # Evitar divisão por zero caso mcap seja 0
    mcap_pct = mcap_df.div(mcap_df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    
    return count_pct.fillna(0), mcap_pct.fillna(0)

# 3.1 Ações vs Não-Ações
count_pct, mcap_pct = analyze_composition(df_hcm, 'year', 'Main_Type')
display_styled_table(count_pct, "Proporção de Ações vs Não-Ações por ANO (Percentual de QUANTIDADE)")
display_styled_table(mcap_pct, "Proporção de Ações vs Não-Ações por ANO (Percentual de MARKET CAP)")



--- Proporção de Ações vs Não-Ações por ANO (Percentual de QUANTIDADE) ---


Main_Type,Non-Stock,Stock
year,,
2014,35.00%,65.00%
2015,36.67%,63.33%
2016,38.24%,61.76%
2017,39.03%,60.97%
2018,40.12%,59.88%
2019,40.42%,59.58%
2020,41.24%,58.76%
2021,42.24%,57.76%
2022,43.19%,56.81%



--- Proporção de Ações vs Não-Ações por ANO (Percentual de MARKET CAP) ---


Main_Type,Non-Stock,Stock
year,,
2014,18.20%,81.80%
2015,17.34%,82.66%
2016,18.88%,81.12%
2017,18.76%,81.24%
2018,19.06%,80.94%
2019,18.96%,81.04%
2020,16.15%,83.85%
2021,16.42%,83.58%
2022,17.11%,82.89%


### 3.2 Composição Detalhada dos Não-Ações


In [38]:
non_stocks = df_hcm[df_hcm['Main_Type'] == 'Non-Stock']
count_pct, mcap_pct = analyze_composition(non_stocks, 'year', 'Asset_Class')

display_styled_table(count_pct, "Tipos de Não-Ações por ANO (Percentual de QUANTIDADE)")
display_styled_table(mcap_pct, "Tipos de Não-Ações por ANO (Percentual de MARKET CAP)")



--- Tipos de Não-Ações por ANO (Percentual de QUANTIDADE) ---


Asset_Class,Closed-End Fund,ETF,REIT
year,,,
2014,26.23%,63.06%,10.70%
2015,24.37%,65.48%,10.16%
2016,23.15%,67.13%,9.72%
2017,22.51%,68.34%,9.15%
2018,21.48%,69.03%,9.50%
2019,20.39%,70.43%,9.18%
2020,19.04%,71.78%,9.18%
2021,17.87%,73.48%,8.64%
2022,16.51%,75.12%,8.36%



--- Tipos de Não-Ações por ANO (Percentual de MARKET CAP) ---


Asset_Class,Closed-End Fund,ETF,REIT
year,,,
2014,8.50%,76.67%,14.83%
2015,4.13%,78.61%,17.25%
2016,3.69%,80.78%,15.52%
2017,3.35%,82.40%,14.25%
2018,2.98%,82.56%,14.46%
2019,2.59%,82.74%,14.67%
2020,2.61%,84.65%,12.74%
2021,2.15%,82.55%,15.30%
2022,2.09%,83.91%,14.00%


### 3.3 Composição do Universo de Ações por Setor


In [39]:
stocks = df_hcm[df_hcm['Main_Type'] == 'Stock']
count_pct, mcap_pct = analyze_composition(stocks, 'year', 'Sector')

display_styled_table(count_pct, "Ações por Setor e ANO (Percentual de QUANTIDADE)")
display_styled_table(mcap_pct, "Ações por Setor e ANO (Percentual de MARKET CAP)")



--- Ações por Setor e ANO (Percentual de QUANTIDADE) ---


Sector,Basic Materials,Communication Services,Consumer Cyclical,Consumer Defensive,Energy,Financial,Healthcare,Industrials,Real Estate,Technology,Utilities
year,,,,,,,,,,,
2014,4.92%,3.33%,12.32%,5.37%,4.97%,21.02%,12.99%,17.29%,0.96%,13.50%,3.33%
2015,4.90%,3.32%,12.36%,5.28%,4.90%,21.13%,13.07%,17.10%,0.98%,13.73%,3.21%
2016,5.01%,3.46%,12.25%,5.27%,5.17%,21.10%,13.16%,16.94%,0.96%,13.48%,3.20%
2017,4.79%,3.56%,12.07%,5.19%,5.60%,20.98%,13.54%,16.70%,0.92%,13.59%,3.05%
2018,4.70%,3.67%,12.24%,5.14%,5.68%,20.70%,14.19%,16.30%,0.98%,13.46%,2.94%
2019,4.61%,3.74%,12.08%,4.89%,5.44%,20.57%,15.36%,15.96%,0.97%,13.51%,2.86%
2020,4.50%,3.88%,11.91%,4.85%,5.29%,20.25%,16.28%,15.62%,1.01%,13.63%,2.78%
2021,4.49%,3.93%,11.79%,4.87%,5.30%,19.91%,16.79%,15.38%,0.98%,13.80%,2.78%
2022,4.44%,4.04%,11.78%,4.81%,5.50%,19.65%,17.28%,15.08%,1.02%,13.74%,2.65%



--- Ações por Setor e ANO (Percentual de MARKET CAP) ---


Sector,Basic Materials,Communication Services,Consumer Cyclical,Consumer Defensive,Energy,Financial,Healthcare,Industrials,Real Estate,Technology,Utilities
year,,,,,,,,,,,
2014,1.92%,19.47%,8.21%,8.78%,6.73%,17.31%,12.38%,9.18%,0.14%,12.91%,2.98%
2015,1.79%,18.06%,10.58%,9.24%,5.25%,17.49%,12.70%,8.79%,0.17%,12.97%,2.95%
2016,2.08%,15.47%,9.72%,9.38%,6.51%,18.87%,11.21%,9.64%,0.14%,13.85%,3.13%
2017,2.14%,15.34%,9.90%,9.36%,5.98%,18.48%,10.92%,9.63%,0.16%,15.27%,2.81%
2018,1.81%,17.40%,10.20%,8.81%,5.30%,16.91%,12.76%,8.29%,0.17%,15.30%,3.05%
2019,1.91%,14.85%,10.32%,8.13%,4.44%,17.22%,12.67%,8.64%,0.21%,18.52%,3.09%
2020,1.68%,23.50%,12.24%,6.97%,2.20%,13.15%,10.82%,7.15%,0.18%,19.88%,2.23%
2021,1.82%,18.12%,12.16%,6.91%,3.03%,13.84%,11.18%,7.36%,0.20%,23.18%,2.21%
2022,2.05%,12.48%,9.11%,8.55%,5.39%,15.99%,13.93%,8.20%,0.21%,20.94%,3.13%


### 3.4 Indústria por setores mais representados

In [54]:
# 1. Obtém os setores já ordenados do mais representado para o menos representado no histórico
sectors = stocks['Sector'].value_counts().index

for sector in sectors:
    sector_df = stocks[stocks["Sector"] == sector].copy()
    
    if sector_df.empty:
        continue
        
    # 2. Encontra as 5 indústrias mais representadas no setor (já vem na ordem decrescente)
    top_5_industries = sector_df['Industry'].value_counts().nlargest(5).index
    
    # 3. Cria a nova coluna onde o que não é Top 5 vira 'Others'
    sector_df['Industry_Grouped'] = sector_df['Industry'].apply(
        lambda x: x if x in top_5_industries else 'Others'
    )
    
    # 4. Executa a análise de proporção
    count_pct, mcap_pct = analyze_composition(sector_df, 'year', 'Industry_Grouped')
    
    # 5. Define a ordem desejada das colunas (Top 5 na ordem de tamanho + 'Others' no final)
    col_order = [ind for ind in top_5_industries if ind in count_pct.columns]
    if 'Others' in count_pct.columns:
        col_order.append('Others')
        
    # Aplica a ordenação das colunas nas duas tabelas
    count_pct = count_pct[col_order]
    mcap_pct = mcap_pct[col_order]
    
    # 6. Exibe as tabelas
    display_styled_table(
        count_pct, 
        f"Ações por Indústria e ANO (Setor: {sector}) - Percentual de QUANTIDADE"
    )
    
    display_styled_table(
        mcap_pct, 
        f"Ações por Indústria e ANO (Setor: {sector}) - Percentual de MARKET CAP"
    )


--- Ações por Indústria e ANO (Setor: Financial) - Percentual de QUANTIDADE ---


Industry_Grouped,Banks - Regional,Asset Management,Capital Markets,Insurance - Property & Casualty,Credit Services,Others
year,,,,,,
2014,56.99%,10.75%,5.91%,7.53%,4.84%,13.98%
2015,55.41%,11.34%,6.19%,7.22%,5.15%,14.69%
2016,54.55%,12.63%,6.31%,7.07%,5.05%,14.39%
2017,54.37%,13.35%,6.55%,6.80%,5.10%,13.83%
2018,53.19%,13.95%,6.62%,6.62%,5.20%,14.42%
2019,51.57%,15.25%,6.73%,6.73%,5.83%,13.90%
2020,50.98%,15.03%,7.63%,6.54%,6.10%,13.73%
2021,50.86%,15.02%,7.94%,6.65%,6.01%,13.52%
2022,51.45%,14.94%,7.68%,6.64%,5.81%,13.49%



--- Ações por Indústria e ANO (Setor: Financial) - Percentual de MARKET CAP ---


Industry_Grouped,Banks - Regional,Asset Management,Capital Markets,Insurance - Property & Casualty,Credit Services,Others
year,,,,,,
2014,10.75%,8.40%,6.33%,4.58%,11.93%,58.02%
2015,11.46%,7.72%,6.22%,4.76%,12.24%,57.60%
2016,13.27%,7.17%,6.50%,4.63%,10.78%,57.65%
2017,11.96%,7.59%,6.34%,4.61%,12.10%,57.40%
2018,11.21%,6.40%,5.30%,4.79%,14.67%,57.63%
2019,12.00%,7.14%,5.08%,4.37%,17.17%,54.24%
2020,9.90%,7.84%,7.05%,4.25%,22.92%,48.04%
2021,10.84%,9.52%,8.15%,3.91%,18.19%,49.41%
2022,10.46%,8.28%,6.55%,4.74%,18.31%,51.66%



--- Ações por Indústria e ANO (Setor: Industrials) - Percentual de QUANTIDADE ---


...

Industry_Grouped,Utilities - Regulated Electric,Utilities - Regulated Gas,Utilities - Regulated Water,Utilities - Diversified,Utilities - Renewable,Others
year,,,,,,
2014,82.72%,5.39%,2.94%,6.29%,1.23%,1.43%
2015,82.42%,5.72%,3.41%,5.44%,2.39%,0.62%
2016,83.07%,6.26%,3.59%,5.29%,1.22%,0.57%
2017,82.03%,6.23%,4.09%,5.05%,1.40%,1.20%
2018,82.09%,6.55%,3.89%,5.45%,0.62%,1.40%
2019,81.46%,6.46%,4.41%,6.23%,0.44%,0.99%
2020,81.74%,5.40%,5.10%,5.82%,1.01%,0.93%
2021,80.76%,5.53%,5.47%,5.42%,1.01%,1.81%
2022,79.82%,5.29%,3.56%,4.66%,5.22%,1.45%



--- Ações por Indústria e ANO (Setor: Real Estate) - Percentual de QUANTIDADE ---


Industry_Grouped,Real Estate Services,Real Estate - Diversified,Real Estate - Development
year,,,
2014,76.47%,11.76%,11.76%
2015,72.22%,16.67%,11.11%
2016,72.22%,16.67%,11.11%
2017,72.22%,16.67%,11.11%
2018,75.00%,15.00%,10.00%
2019,71.43%,14.29%,14.29%
2020,73.91%,13.04%,13.04%
2021,73.91%,13.04%,13.04%
2022,72.00%,12.00%,16.00%



--- Ações por Indústria e ANO (Setor: Real Estate) - Percentual de MARKET CAP ---


Industry_Grouped,Real Estate Services,Real Estate - Diversified,Real Estate - Development
year,,,
2014,92.03%,6.13%,1.83%
2015,81.77%,17.08%,1.16%
2016,78.48%,19.82%,1.70%
2017,81.78%,16.00%,2.22%
2018,86.54%,12.00%,1.46%
2019,87.88%,9.17%,2.95%
2020,88.25%,8.68%,3.07%
2021,87.44%,8.71%,3.85%
2022,88.67%,7.78%,3.55%


In [48]:
fin_sector = stocks[stocks["Sector"]=="Financial"]

count_pct, mcap_pct = analyze_composition(fin_sector, 'year', 'Industry')

display_styled_table(count_pct, "Ações por Indústria e ANO (Setor Financials) (Percentual de QUANTIDADE)")
display_styled_table(mcap_pct, "Ações por Setor e ANO (Setor Financials) (Percentual de MARKET CAP)")


--- Ações por Indústria e ANO (Setor Financials) (Percentual de QUANTIDADE) ---


Industry,Asset Management,Banks - Diversified,Banks - Regional,Capital Markets,Credit Services,Financial Conglomerates,Financial Data & Stock Exchanges,Insurance - Diversified,Insurance - Life,Insurance - Property & Casualty,Insurance - Reinsurance,Insurance - Specialty,Insurance Brokers,Mortgage Finance
year,,,,,,,,,,,,,,
2014,10.75%,1.61%,56.99%,5.91%,4.84%,0.81%,2.42%,1.08%,2.96%,7.53%,0.27%,1.88%,2.15%,0.81%
2015,11.34%,1.55%,55.41%,6.19%,5.15%,0.77%,2.58%,1.03%,3.09%,7.22%,0.26%,2.32%,2.06%,1.03%
2016,12.63%,1.52%,54.55%,6.31%,5.05%,0.76%,2.53%,1.01%,3.03%,7.07%,0.25%,2.27%,2.02%,1.01%
2017,13.35%,1.46%,54.37%,6.55%,5.10%,0.73%,2.43%,0.97%,2.91%,6.80%,0.24%,2.18%,1.94%,0.97%
2018,13.95%,1.42%,53.19%,6.62%,5.20%,0.95%,2.36%,0.95%,2.84%,6.62%,0.24%,2.60%,1.89%,1.18%
2019,15.25%,1.35%,51.57%,6.73%,5.83%,0.90%,2.24%,0.90%,2.69%,6.73%,0.22%,2.69%,1.79%,1.12%
2020,15.03%,1.31%,50.98%,7.63%,6.10%,0.87%,2.40%,0.87%,2.61%,6.54%,0.22%,2.61%,1.74%,1.09%
2021,15.02%,1.29%,50.86%,7.94%,6.01%,0.86%,2.36%,0.86%,2.58%,6.65%,0.21%,2.58%,1.72%,1.07%
2022,14.94%,1.24%,51.45%,7.68%,5.81%,0.83%,2.28%,0.83%,2.70%,6.64%,0.21%,2.49%,1.66%,1.24%



--- Ações por Setor e ANO (Setor Financials) (Percentual de MARKET CAP) ---


Industry,Asset Management,Banks - Diversified,Banks - Regional,Capital Markets,Credit Services,Financial Conglomerates,Financial Data & Stock Exchanges,Insurance - Diversified,Insurance - Life,Insurance - Property & Casualty,Insurance - Reinsurance,Insurance - Specialty,Insurance Brokers,Mortgage Finance
year,,,,,,,,,,,,,,
2014,8.40%,25.76%,10.75%,6.33%,11.93%,0.07%,3.26%,22.59%,4.35%,4.58%,0.16%,0.45%,1.36%,0.01%
2015,7.72%,25.87%,11.46%,6.22%,12.24%,0.09%,3.96%,21.51%,4.05%,4.76%,0.16%,0.60%,1.33%,0.03%
2016,7.17%,25.81%,13.27%,6.50%,10.78%,0.11%,3.75%,21.69%,4.09%,4.63%,0.20%,0.58%,1.39%,0.03%
2017,7.59%,25.73%,11.96%,6.34%,12.10%,0.14%,4.28%,21.26%,3.76%,4.61%,0.20%,0.61%,1.39%,0.04%
2018,6.40%,22.28%,11.21%,5.30%,14.67%,0.24%,5.22%,24.17%,3.24%,4.79%,0.20%,0.60%,1.62%,0.08%
2019,7.14%,21.96%,12.00%,5.08%,17.17%,0.28%,5.49%,21.18%,2.92%,4.37%,0.18%,0.66%,1.46%,0.09%
2020,7.84%,16.95%,9.90%,7.05%,22.92%,0.25%,6.44%,19.24%,2.37%,4.25%,0.14%,0.55%,1.97%,0.13%
2021,9.52%,17.26%,10.84%,8.15%,18.19%,0.19%,7.20%,19.44%,2.35%,3.91%,0.10%,0.54%,2.20%,0.13%
2022,8.28%,14.31%,10.46%,6.55%,18.31%,0.06%,6.18%,24.75%,3.06%,4.74%,0.16%,0.46%,2.67%,0.01%


## 4. Comparação: Decil 1 (Central) vs Decil 10 (Periférico)


In [40]:
def extract_decile_analysis(df):
    df['decile_num'] = df['portfolio'].str.extract(r'(\d+)').astype(float)
    max_decile_val = int(df['decile_num'].max())
    max_decile_str = f'decil_{max_decile_val}'
    
    d1 = df[df['portfolio'] == 'decil_1'].copy()
    d10 = df[df['portfolio'] == max_decile_str].copy()
    
    d1['Grupo'] = 'Decil 1 (Central)'
    d10['Grupo'] = f'Decil {max_decile_val} (Periférico)'
    
    return pd.concat([d1, d10])

df_hcm_comp = extract_decile_analysis(df_hcm)
df_pozzi_comp = extract_decile_analysis(df_pozzi)


### 4.1 HCM: Decil Central vs Periférico - Tipo de Ativo


In [41]:
def display_decile_comparison(df_comp, metric_name):
    # Agregar por Ano, Grupo e Classe de Ativo
    count_df = df_comp.groupby(['year', 'Grupo', 'Asset_Class']).size().unstack(fill_value=0)
    count_pct = count_df.div(count_df.sum(axis=1), axis=0) * 100
    
    mcap_df = df_comp.groupby(['year', 'Grupo', 'Asset_Class'])['mcap'].sum().unstack(fill_value=0)
    mcap_pct = mcap_df.div(mcap_df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    
    display_styled_table(count_pct.fillna(0), f"{metric_name} - Composição de Ativos (Percentual de QUANTIDADE)")
    display_styled_table(mcap_pct.fillna(0), f"{metric_name} - Composição de Ativos (Percentual de MARKET CAP)")

display_decile_comparison(df_hcm_comp, "HCM")



--- HCM - Composição de Ativos (Percentual de QUANTIDADE) ---



--- HCM - Composição de Ativos (Percentual de MARKET CAP) ---


### 4.2 Pozzi: Decil Central vs Periférico - Tipo de Ativo


In [42]:
display_decile_comparison(df_pozzi_comp, "Pozzi")



--- Pozzi - Composição de Ativos (Percentual de QUANTIDADE) ---



--- Pozzi - Composição de Ativos (Percentual de MARKET CAP) ---


### 4.3 Composição Setorial das Ações (Decil 1 vs Decil Periférico)


In [43]:
def display_sector_comparison(df_comp, metric_name):
    stocks_only = df_comp[df_comp['Main_Type'] == 'Stock']
    
    count_df = stocks_only.groupby(['year', 'Grupo', 'Sector']).size().unstack(fill_value=0)
    count_pct = count_df.div(count_df.sum(axis=1), axis=0) * 100
    
    mcap_df = stocks_only.groupby(['year', 'Grupo', 'Sector'])['mcap'].sum().unstack(fill_value=0)
    mcap_pct = mcap_df.div(mcap_df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    
    display_styled_table(count_pct.fillna(0), f"{metric_name} - Setores das Ações (Percentual de QUANTIDADE)")
    display_styled_table(mcap_pct.fillna(0), f"{metric_name} - Setores das Ações (Percentual de MARKET CAP)")

display_sector_comparison(df_hcm_comp, "HCM")
display_sector_comparison(df_pozzi_comp, "Pozzi")



--- HCM - Setores das Ações (Percentual de QUANTIDADE) ---



--- HCM - Setores das Ações (Percentual de MARKET CAP) ---



--- Pozzi - Setores das Ações (Percentual de QUANTIDADE) ---



--- Pozzi - Setores das Ações (Percentual de MARKET CAP) ---


### Composição industrial das ações

In [60]:
def display_top5_industries_ranking(df_comp, metric_name):
    # Função interna para pegar o Top 5 formatado
    def get_top_5(year, decil):
        # AQUI FOI CORRIGIDO: usando 'portfolio' e passando o nome exato do decil
        sub_df = df_comp[(df_comp['year'] == str(year)) & 
                         (df_comp['portfolio'] == decil) & 
                         (df_comp['Main_Type'] == 'Stock')]
        
        result = []
        if not sub_df.empty:
            # Pegamos os 5 mais presentes e transformamos em % 
            top5 = sub_df['Industry'].value_counts(normalize=True).head(5)
            result = [f"{nome} ({val*100:.1f}%)" for nome, val in top5.items()]
            
        while len(result) < 5:
            result.append("-")
            
        return result

    years = sorted(df_comp['year'].dropna().unique())
    d1_data, d10_data = [], []
    
    # Processa ano a ano apontando para 'decil_1' e 'decil_10'
    for y in years:
        d1_data.append([y] + get_top_5(y, 'decil_1'))
        d10_data.append([y] + get_top_5(y, 'decil_10'))
        
    cols = ['Year', 'Top 1', 'Top 2', 'Top 3', 'Top 4', 'Top 5']
    d1_df = pd.DataFrame(d1_data, columns=cols).set_index('Year')
    d10_df = pd.DataFrame(d10_data, columns=cols).set_index('Year')
    
    # Exibe as duas tabelas
    print(f"\n{'='*60}\n {metric_name} - NÚCLEO (DECIL 1) - TOP 5 INDÚSTRIAS\n{'='*60}")
    display(d1_df)
    
    print(f"\n{'='*60}\n {metric_name} - PERIFERIA (DECIL 10) - TOP 5 INDÚSTRIAS\n{'='*60}")
    display(d10_df)

# Basta chamar para as duas métricas (usando df_hcm e df_pozzi diretamente):
display_top5_industries_ranking(df_hcm, "HCM")
display_top5_industries_ranking(df_pozzi, "POZZI")


 HCM - NÚCLEO (DECIL 1) - TOP 5 INDÚSTRIAS


,Top 1,Top 2,Top 3,Top 4,Top 5
Year,,,,,
2014,Banks - Regional (15.1%),Specialty Industrial Machinery (6.3%),Specialty Chemicals (4.4%),Engineering & Construction (3.9%),Electronic Components (3.4%)
2015,Banks - Regional (20.6%),Software - Application (6.1%),Engineering & Construction (4.2%),Auto Parts (3.3%),Building Products & Equipment (2.8%)
2016,Banks - Regional (18.6%),Specialty Industrial Machinery (5.2%),Software - Application (4.6%),Asset Management (4.1%),Aerospace & Defense (4.1%)
2017,Specialty Industrial Machinery (7.3%),Asset Management (5.7%),Software - Infrastructure (5.7%),Banks - Regional (4.9%),Auto Parts (4.9%)
2018,Banks - Regional (20.1%),Specialty Industrial Machinery (5.3%),Engineering & Construction (4.3%),Aerospace & Defense (3.8%),Specialty Chemicals (3.8%)
2019,Banks - Regional (35.1%),Specialty Industrial Machinery (7.8%),Engineering & Construction (3.7%),Credit Services (2.9%),Metal Fabrication (2.9%)
2020,Specialty Industrial Machinery (9.3%),Asset Management (6.0%),Banks - Regional (6.0%),Building Products & Equipment (4.0%),Steel (3.3%)
2021,Banks - Regional (8.2%),Specialty Industrial Machinery (7.1%),Asset Management (5.3%),Specialty Chemicals (4.1%),Aerospace & Defense (3.5%)
2022,Specialty Industrial Machinery (9.7%),Asset Management (7.8%),Specialty Chemicals (7.1%),Banks - Regional (5.8%),Building Products & Equipment (5.2%)



 HCM - PERIFERIA (DECIL 10) - TOP 5 INDÚSTRIAS


,Top 1,Top 2,Top 3,Top 4,Top 5
Year,,,,,
2014,Banks - Regional (26.6%),Biotechnology (8.1%),Semiconductors (8.1%),Oil & Gas E&P (4.0%),Gold (3.2%)
2015,Banks - Regional (23.0%),Utilities - Regulated Electric (7.7%),Biotechnology (7.7%),Gold (3.1%),Medical Devices (2.6%)
2016,Banks - Regional (18.3%),Utilities - Regulated Electric (10.4%),Biotechnology (4.9%),Asset Management (3.7%),Capital Markets (3.0%)
2017,Banks - Regional (19.3%),Utilities - Regulated Electric (15.2%),Biotechnology (5.5%),Capital Markets (3.4%),Real Estate Services (2.8%)
2018,Utilities - Regulated Electric (16.4%),Banks - Regional (13.2%),Biotechnology (9.2%),Software - Application (5.3%),Oil & Gas E&P (3.3%)
2019,Utilities - Regulated Electric (15.8%),Biotechnology (10.2%),Banks - Regional (10.2%),Capital Markets (5.6%),Asset Management (3.4%)
2020,Asset Management (12.0%),Banks - Regional (8.9%),Biotechnology (6.3%),Software - Application (3.8%),Software - Infrastructure (3.2%)
2021,Asset Management (10.6%),Restaurants (7.6%),Banks - Regional (7.1%),Oil & Gas E&P (5.3%),Software - Application (4.1%)
2022,Biotechnology (34.0%),Asset Management (11.7%),Drug Manufacturers - Specialty & Generic (3.7%),Banks - Regional (3.7%),Packaged Foods (3.2%)



 POZZI - NÚCLEO (DECIL 1) - TOP 5 INDÚSTRIAS


,Top 1,Top 2,Top 3,Top 4,Top 5
Year,,,,,
2014,Asset Management (5.6%),Diagnostics & Research (4.9%),Aerospace & Defense (4.9%),Packaged Foods (4.2%),Healthcare Plans (4.2%)
2015,Banks - Regional (12.6%),Insurance - Property & Casualty (11.9%),Asset Management (10.4%),Financial Data & Stock Exchanges (4.4%),Capital Markets (4.4%)
2016,Oil & Gas E&P (11.5%),Oil & Gas Equipment & Services (8.3%),Banks - Regional (7.4%),Specialty Chemicals (4.1%),Oil & Gas Midstream (4.1%)
2017,Oil & Gas Equipment & Services (8.4%),Oil & Gas E&P (7.3%),Asset Management (5.2%),Oil & Gas Midstream (4.2%),Chemicals (3.7%)
2018,Oil & Gas E&P (6.6%),Asset Management (5.2%),Oil & Gas Equipment & Services (5.2%),Oil & Gas Midstream (3.9%),Specialty Chemicals (3.9%)
2019,Banks - Regional (8.3%),Insurance - Property & Casualty (6.2%),Trucking (5.5%),Specialty Chemicals (5.5%),Asset Management (4.8%)
2020,Banks - Regional (10.4%),Asset Management (5.8%),Insurance - Property & Casualty (5.4%),Aerospace & Defense (5.0%),Capital Markets (5.0%)
2021,Banks - Regional (8.8%),Insurance - Property & Casualty (7.5%),Oil & Gas E&P (5.7%),Capital Markets (5.3%),Aerospace & Defense (4.4%)
2022,Credit Services (7.0%),Capital Markets (7.0%),Diagnostics & Research (6.4%),Insurance - Property & Casualty (5.2%),Drug Manufacturers - General (4.7%)



 POZZI - PERIFERIA (DECIL 10) - TOP 5 INDÚSTRIAS


,Top 1,Top 2,Top 3,Top 4,Top 5
Year,,,,,
2014,Banks - Regional (15.5%),Semiconductors (11.3%),Oil & Gas E&P (8.2%),Gold (7.2%),Biotechnology (6.2%)
2015,Biotechnology (20.6%),Banks - Regional (15.0%),Gold (3.7%),Healthcare Plans (2.8%),Diagnostics & Research (2.8%)
2016,Banks - Regional (18.9%),Trucking (3.4%),Medical Instruments & Supplies (3.0%),Residential Construction (3.0%),Biotechnology (3.0%)
2017,Banks - Regional (15.4%),Utilities - Regulated Electric (13.6%),Biotechnology (9.3%),Software - Application (3.7%),Medical Devices (3.1%)
2018,Biotechnology (25.2%),Semiconductors (10.7%),Semiconductor Equipment & Materials (6.1%),Drug Manufacturers - Specialty & Generic (5.3%),Banks - Regional (4.2%)
2019,Biotechnology (11.0%),Banks - Regional (10.3%),Apparel Retail (5.1%),Specialty Retail (4.8%),Software - Application (4.0%)
2020,Asset Management (13.6%),Banks - Regional (7.3%),Biotechnology (5.1%),Credit Services (3.4%),Airlines (3.4%)
2021,Restaurants (9.8%),Asset Management (8.4%),Banks - Regional (5.6%),Software - Application (4.2%),Travel Services (4.2%)
2022,Biotechnology (50.8%),Asset Management (8.4%),Drug Manufacturers - Specialty & Generic (4.7%),Restaurants (3.7%),Medical Devices (2.7%)
